# 04 Device-directed speech detection with leave-corpus-out evaluation and shortcut probes

In [ ]:
# Settings: GPU T4 x2, Internet ON. Persistent outputs: results/ and runs/ in /kaggle/working.
import os, shutil, subprocess
REPO_URL = "https://github.com/ByteBoss123/speaker-aware-speech-llm.git"
WORK = "/kaggle/working/speaker-aware-speech-llm"
if not os.path.exists(WORK):
    src = "/kaggle/input/speaker-aware-speech-llm"   # alternative: upload the repo as a Kaggle dataset
    if os.path.exists(src):
        shutil.copytree(src, WORK)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
os.chdir(WORK)
DATA = "/tmp/data"   # large ephemeral disk; /kaggle/working is capped at ~20 GB
os.environ["DATA"] = DATA

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | tail -2

In [ ]:
!bash scripts/prepare_data.sh $DATA slurp vox1test

Export SLURP + MInDS-14 (directed) and AMI + VoxCeleb1 (non-directed) to 16 kHz wav

In [ ]:
!python scripts/ddsd_prepare.py --data_root $DATA --out $DATA/ddsd

In [ ]:
!python scripts/ddsd_features.py --ddsd_dir $DATA/ddsd --out results/ddsd

A model only counts as learning directedness where it beats the channel-only and length-only probes cross-corpus

In [ ]:
!python scripts/ddsd_eval.py --feat_dir results/ddsd

In [ ]:
!python scripts/summarize.py results runs
!cd /kaggle/working && zip -qr results_ddsd.zip speaker-aware-speech-llm/results speaker-aware-speech-llm/runs -x '*.npz'